In [1]:
import pandas as pd
import duckdb
import os

raw_path = "../data/raw/tmall_order_report.csv"

df = pd.read_csv(raw_path)

print(df.shape)
df.head()

(28010, 7)


,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8


In [3]:
import pandas as pd
import duckdb
import os

raw_path = "../data/raw/tmall_order_report.csv"

df = pd.read_csv(raw_path)

# 去掉字段名前后的空格，防止“收货地址 ”这种隐藏空格报错
df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

print(df.shape)
print(df.columns.tolist())

df.head()

(28010, 7)
['订单编号', '总金额', '买家实际支付金额', '收货地址', '订单创建时间', '订单付款时间', '退款金额']


,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8


In [ ]:
import duckdb
con = duckdb.connect("../tmall_project.duckdb")
con.register("raw_orders", df)
# 用 SQL 清洗原始订单数据，并生成 clean_orders 表
con.execute("""
CREATE OR REPLACE TABLE clean_orders AS
SELECT
    CAST("订单编号" AS BIGINT) AS order_id,
    CAST("总金额" AS DOUBLE) AS total_amount,
    CAST("买家实际支付金额" AS DOUBLE) AS pay_amount,
    "收货地址" AS province,
    CAST("订单创建时间" AS TIMESTAMP) AS order_create_time,
    CAST("订单付款时间" AS TIMESTAMP) AS pay_time,
    CAST("退款金额" AS DOUBLE) AS refund_amount,
    CAST("订单创建时间" AS DATE) AS order_date,

    CASE
        WHEN CAST("退款金额" AS DOUBLE) > 0 THEN '退款订单'
        WHEN "订单付款时间" IS NULL OR CAST("买家实际支付金额" AS DOUBLE) = 0 THEN '未支付订单'
        ELSE '已支付订单'
    END AS order_status

FROM raw_orders
WHERE "订单编号" IS NOT NULL;
""")
# 查看清洗后的数据
clean_df = con.execute("""
SELECT *
FROM clean_orders
LIMIT 10;
""").df()

clean_df

,order_id,total_amount,pay_amount,province,order_create_time,pay_time,refund_amount,order_date,order_status
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaT,0.0,2020-02-21,未支付订单
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0,2020-02-20,已支付订单
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaT,0.0,2020-02-20,未支付订单
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0,2020-02-20,已支付订单
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8,2020-02-20,退款订单
5,6,327.7,148.9,浙江省,2020-02-20 23:56:39,2020-02-20 23:56:53,178.8,2020-02-20,退款订单
6,7,357.0,357.0,天津,2020-02-20 23:56:36,2020-02-20 23:56:40,0.0,2020-02-20,已支付订单
7,8,53.0,53.0,浙江省,2020-02-20 23:56:12,2020-02-20 23:56:16,0.0,2020-02-20,已支付订单
8,9,43.0,0.0,湖南省,2020-02-20 23:54:53,2020-02-20 23:55:04,43.0,2020-02-20,退款订单
9,10,421.0,421.0,北京,2020-02-20 23:54:28,2020-02-20 23:54:33,0.0,2020-02-20,已支付订单


In [5]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS order_cnt,
    SUM(total_amount) AS total_amount,
    SUM(pay_amount) AS pay_amount,
    SUM(refund_amount) AS refund_amount
FROM clean_orders;
""").df()

,total_rows,order_cnt,total_amount,pay_amount,refund_amount
0,28010,28010,2995760.61,1902487.15,572335.92


In [6]:
con.execute("""
SELECT
    order_status,
    COUNT(*) AS order_cnt,
    SUM(total_amount) AS total_amount,
    SUM(pay_amount) AS pay_amount,
    SUM(refund_amount) AS refund_amount
FROM clean_orders
GROUP BY order_status
ORDER BY order_cnt DESC;
""").df()

,order_status,order_cnt,total_amount,pay_amount,refund_amount
0,已支付订单,18441,1861091.01,1861091.01,0.00
1,退款订单,5646,613732.06,41396.14,572335.92
2,未支付订单,3923,520937.54,0.00,0.00


In [7]:
# 02 核心经营指标分析
# 1. 整体核心指标
overall_indicators = con.execute("""
SELECT
    COUNT(DISTINCT order_id) AS total_order_cnt,
    COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) AS paid_order_cnt,
    COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) AS refund_order_cnt,

    ROUND(SUM(total_amount), 2) AS total_gmv,
    ROUND(SUM(pay_amount), 2) AS paid_amount,
    ROUND(SUM(refund_amount), 2) AS refund_amount,
    ROUND(SUM(pay_amount) - SUM(refund_amount), 2) AS net_paid_amount,

    ROUND(AVG(CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN pay_amount END), 2) AS avg_order_value,

    ROUND(
        COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) * 1.0 
        / COUNT(DISTINCT order_id), 
        4
    ) AS pay_order_rate,

    ROUND(
        COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) * 1.0 
        / COUNT(DISTINCT order_id), 
        4
    ) AS refund_order_rate,

    ROUND(
        SUM(refund_amount) / NULLIF(SUM(pay_amount), 0), 
        4
    ) AS refund_amount_rate

FROM clean_orders;
""").df()

overall_indicators

,total_order_cnt,paid_order_cnt,refund_order_cnt,total_gmv,paid_amount,refund_amount,net_paid_amount,avg_order_value,pay_order_rate,refund_order_rate,refund_amount_rate
0,28010,18955,5646,2995760.61,1902487.15,572335.92,1330151.23,100.37,0.6767,0.2016,0.3008


In [8]:
# 2. 每日经营指标趋势
daily_indicators = con.execute("""
SELECT
    order_date,
    COUNT(DISTINCT order_id) AS total_order_cnt,
    COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) AS paid_order_cnt,
    COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) AS refund_order_cnt,

    ROUND(SUM(total_amount), 2) AS total_gmv,
    ROUND(SUM(pay_amount), 2) AS paid_amount,
    ROUND(SUM(refund_amount), 2) AS refund_amount,
    ROUND(SUM(pay_amount) - SUM(refund_amount), 2) AS net_paid_amount,

    ROUND(AVG(CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN pay_amount END), 2) AS avg_order_value,

    ROUND(
        COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) * 1.0
        / COUNT(DISTINCT order_id),
        4
    ) AS pay_order_rate,

    ROUND(
        COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) * 1.0
        / COUNT(DISTINCT order_id),
        4
    ) AS refund_order_rate

FROM clean_orders
GROUP BY order_date
ORDER BY order_date;
""").df()

daily_indicators.head()

,order_date,total_order_cnt,paid_order_cnt,refund_order_cnt,total_gmv,paid_amount,refund_amount,net_paid_amount,avg_order_value,pay_order_rate,refund_order_rate
0,2020-02-01,176,87,78,16673.00,7031.00,7514.00,-483.00,80.82,0.4943,0.4432
1,2020-02-02,222,107,98,23919.00,8508.00,12429.00,-3921.00,79.51,0.4820,0.4414
2,2020-02-03,267,136,107,22357.00,11316.00,8115.00,3201.00,83.21,0.5094,0.4007
3,2020-02-04,469,254,169,40841.89,21926.74,14730.98,7195.76,86.33,0.5416,0.3603
4,2020-02-05,369,194,140,33205.00,15685.00,14214.00,1471.00,80.85,0.5257,0.3794


In [9]:
# 3. 地区维度分析
province_indicators = con.execute("""
SELECT
    province,
    COUNT(DISTINCT order_id) AS total_order_cnt,
    COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) AS paid_order_cnt,
    COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) AS refund_order_cnt,

    ROUND(SUM(total_amount), 2) AS total_gmv,
    ROUND(SUM(pay_amount), 2) AS paid_amount,
    ROUND(SUM(refund_amount), 2) AS refund_amount,
    ROUND(SUM(pay_amount) - SUM(refund_amount), 2) AS net_paid_amount,

    ROUND(AVG(CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN pay_amount END), 2) AS avg_order_value,

    ROUND(
        COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) * 1.0
        / COUNT(DISTINCT order_id),
        4
    ) AS refund_order_rate

FROM clean_orders
GROUP BY province
ORDER BY paid_amount DESC;
""").df()

province_indicators.head(10)

,province,total_order_cnt,paid_order_cnt,refund_order_cnt,total_gmv,paid_amount,refund_amount,net_paid_amount,avg_order_value,refund_order_rate
0,上海,3353,2470,647,544907.63,264039.78,62418.01,201621.77,106.90,0.1930
1,北京,2054,1489,396,231055.49,166448.48,45941.36,120507.12,111.79,0.1928
2,江苏省,2126,1459,426,227930.93,159359.18,43011.34,116347.84,109.22,0.2004
3,广东省,2463,1585,482,227855.28,147822.90,45588.70,102234.20,93.26,0.1957
4,浙江省,2061,1438,428,203126.96,141664.80,43234.80,98430.00,98.52,0.2077
5,四川省,2019,1380,408,188948.12,127648.15,40299.76,87348.39,92.50,0.2021
6,山东省,1804,1145,380,175046.13,103917.26,45415.21,58502.05,90.76,0.2106
7,天津,1153,838,209,124564.24,89990.06,22761.08,67228.98,107.39,0.1813
8,辽宁省,1187,812,225,107355.93,74692.05,18860.40,55831.65,91.99,0.1896
9,重庆,1036,691,221,108975.65,71514.65,21753.00,49761.65,103.49,0.2133


In [10]:
# 4. 订单状态分析
status_indicators = con.execute("""
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS order_cnt,
    ROUND(SUM(total_amount), 2) AS total_gmv,
    ROUND(SUM(pay_amount), 2) AS paid_amount,
    ROUND(SUM(refund_amount), 2) AS refund_amount,
    ROUND(COUNT(DISTINCT order_id) * 1.0 / SUM(COUNT(DISTINCT order_id)) OVER (), 4) AS order_cnt_ratio
FROM clean_orders
GROUP BY order_status
ORDER BY order_cnt DESC;
""").df()

status_indicators

,order_status,order_cnt,total_gmv,paid_amount,refund_amount,order_cnt_ratio
0,已支付订单,18441,1861091.01,1861091.01,0.00,0.6584
1,退款订单,5646,613732.06,41396.14,572335.92,0.2016
2,未支付订单,3923,520937.54,0.00,0.00,0.1401


In [11]:
# 导出处理后的数据
import os
processed_path = "../data/processed"
os.makedirs(processed_path, exist_ok=True)
# 导出清洗后的明细表
clean_orders_df = con.execute("""
SELECT *
FROM clean_orders;
""").df()
clean_orders_df.to_csv("../data/processed/clean_orders.csv", index=False, encoding="utf-8-sig")
overall_indicators.to_csv("../data/processed/overall_indicators.csv", index=False, encoding="utf-8-sig")
daily_indicators.to_csv("../data/processed/daily_indicators.csv", index=False, encoding="utf-8-sig")
province_indicators.to_csv("../data/processed/province_indicators.csv", index=False, encoding="utf-8-sig")
status_indicators.to_csv("../data/processed/status_indicators.csv", index=False, encoding="utf-8-sig")

print("数据已导出到 data/processed 文件夹")

数据已导出到 data/processed 文件夹


In [14]:
#  04 销售异动分析

abnormal_analysis = con.execute("""
WITH daily AS (
    SELECT
        order_date,
        COUNT(DISTINCT order_id) AS total_order_cnt,
        COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL AND pay_amount > 0 THEN order_id END) AS paid_order_cnt,
        COUNT(DISTINCT CASE WHEN refund_amount > 0 THEN order_id END) AS refund_order_cnt,
        ROUND(SUM(pay_amount), 2) AS paid_amount,
        ROUND(SUM(refund_amount), 2) AS refund_amount,
        ROUND(SUM(pay_amount) - SUM(refund_amount), 2) AS net_paid_amount
    FROM clean_orders
    GROUP BY order_date
),

daily_with_lag AS (
    SELECT
        *,
        LAG(paid_amount) OVER (ORDER BY order_date) AS last_day_paid_amount,
        LAG(total_order_cnt) OVER (ORDER BY order_date) AS last_day_order_cnt,
        LAG(refund_amount) OVER (ORDER BY order_date) AS last_day_refund_amount
    FROM daily
),

daily_calc AS (
    SELECT
        order_date,
        total_order_cnt,
        paid_order_cnt,
        refund_order_cnt,
        paid_amount,
        refund_amount,
        net_paid_amount,

        last_day_paid_amount,
        last_day_order_cnt,
        last_day_refund_amount,

        ROUND(paid_amount - last_day_paid_amount, 2) AS paid_amount_diff,
        total_order_cnt - last_day_order_cnt AS order_cnt_diff,
        ROUND(refund_amount - last_day_refund_amount, 2) AS refund_amount_diff,

        CASE
            WHEN last_day_paid_amount IS NULL OR last_day_paid_amount < 1000 THEN NULL
            ELSE ROUND((paid_amount - last_day_paid_amount) / last_day_paid_amount, 4)
        END AS paid_amount_growth_rate,

        CASE
            WHEN last_day_order_cnt IS NULL OR last_day_order_cnt < 50 THEN NULL
            ELSE ROUND((total_order_cnt - last_day_order_cnt) * 1.0 / last_day_order_cnt, 4)
        END AS order_cnt_growth_rate,

        CASE
            WHEN last_day_refund_amount IS NULL OR last_day_refund_amount < 1000 THEN NULL
            ELSE ROUND((refund_amount - last_day_refund_amount) / last_day_refund_amount, 4)
        END AS refund_amount_growth_rate

    FROM daily_with_lag
)

SELECT
    *,
    CASE
        WHEN paid_amount_growth_rate >= 0.3 AND paid_amount_diff >= 10000 THEN '销售额明显上涨'
        WHEN paid_amount_growth_rate <= -0.3 AND paid_amount_diff <= -10000 THEN '销售额明显下跌'
        WHEN refund_amount_growth_rate >= 0.3 AND refund_amount_diff >= 5000 THEN '退款金额明显上涨'
        ELSE '正常波动'
    END AS abnormal_type

FROM daily_calc
ORDER BY order_date;
""").df()

abnormal_analysis

,order_date,total_order_cnt,paid_order_cnt,refund_order_cnt,paid_amount,refund_amount,net_paid_amount,last_day_paid_amount,last_day_order_cnt,last_day_refund_amount,paid_amount_diff,order_cnt_diff,refund_amount_diff,paid_amount_growth_rate,order_cnt_growth_rate,refund_amount_growth_rate,abnormal_type
0,2020-02-01,176,87,78,7031.00,7514.00,-483.00,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,正常波动
1,2020-02-02,222,107,98,8508.00,12429.00,-3921.00,7031.00,176,7514.00,1477.00,46,4915.00,0.2101,0.2614,0.6541,正常波动
2,2020-02-03,267,136,107,11316.00,8115.00,3201.00,8508.00,222,12429.00,2808.00,45,-4314.00,0.3300,0.2027,-0.3471,正常波动
3,2020-02-04,469,254,169,21926.74,14730.98,7195.76,11316.00,267,8115.00,10610.74,202,6615.98,0.9377,0.7566,0.8153,销售额明显上涨
4,2020-02-05,369,194,140,15685.00,14214.00,1471.00,21926.74,469,14730.98,-6241.74,-100,-516.98,-0.2847,-0.2132,-0.0351,正常波动
5,2020-02-06,144,71,58,5949.00,6131.00,-182.00,15685.00,369,14214.00,-9736.00,-225,-8083.00,-0.6207,-0.6098,-0.5687,正常波动
6,2020-02-07,177,95,62,7109.00,5586.00,1523.00,5949.00,144,6131.00,1160.00,33,-545.00,0.1950,0.2292,-0.0889,正常波动
7,2020-02-09,404,266,87,22123.00,10255.00,11868.00,7109.00,177,5586.00,15014.00,227,4669.00,2.1120,1.2825,0.8358,销售额明显上涨
8,2020-02-10,27,16,8,1010.00,433.00,577.00,22123.00,404,10255.00,-21113.00,-377,-9822.00,-0.9543,-0.9332,-0.9578,销售额明显下跌
9,2020-02-11,15,14,1,364.00,22.00,342.00,1010.00,27,433.00,-646.00,-12,-411.00,-0.6396,NaN,NaN,正常波动


In [15]:
abnormal_analysis.to_csv("../data/processed/abnormal_analysis.csv", index=False, encoding="utf-8-sig")

print("abnormal_analysis.csv 已导出")

abnormal_analysis.csv 已导出
